# 前処理

EDAで特定されたベイスギの異常サンプル（サンプルナンバー 1556〜1607）を除外し、クリーン済みの学習データを保存する。

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/train.csv", encoding="shift-jis")
print(f"読み込み件数: {len(df)}")
df.head()

読み込み件数: 1322


,sample number,species number,樹種,含水率,9993.76781,9989.9107,9986.05359,9982.19648,9978.33937,9974.48227,...,4034.53536,4030.67826,4026.82115,4022.96404,4019.10693,4015.24982,4011.39271,4007.5356,4003.6785,3999.82139
0,1,1,イチョウ,216.129032,0.41485,0.41465,0.41463,0.41476,0.41481,0.41470,...,1.25104,1.24925,1.24145,1.23620,1.23384,1.22981,1.22818,1.23087,1.23354,1.23219
1,2,1,イチョウ,210.752688,0.42049,0.42040,0.42049,0.42053,0.42038,0.42010,...,1.21929,1.21611,1.21565,1.21745,1.21680,1.21205,1.21074,1.21508,1.21901,1.21846
2,3,1,イチョウ,205.913979,0.41040,0.41045,0.41047,0.41028,0.41000,0.40989,...,1.17471,1.17147,1.16611,1.16633,1.16998,1.16955,1.16200,1.15341,1.15139,1.15403
3,4,1,イチョウ,201.075269,0.40080,0.40061,0.40031,0.40008,0.39993,0.39991,...,1.11271,1.11339,1.11115,1.10965,1.11017,1.11165,1.11426,1.11787,1.11849,1.11328
4,5,1,イチョウ,196.236559,0.38792,0.38814,0.38825,0.38817,0.38798,0.38778,...,1.05617,1.05596,1.05760,1.06014,1.06240,1.06566,1.06962,1.07056,1.06459,1.05612


## ベイスギ異常サンプルの除去

サンプルナンバー 1556〜1607 はEDAで異常スペクトル（低含水率域での波数端スパイク）が確認されたため除外する。

In [2]:
# サンプルナンバー 1556〜1607 を除外
mask = ~df["sample number"].between(1556, 1607)
cleaned_beisugi_train = df[mask].reset_index(drop=True)

removed = df[~mask]
print(f"除外件数: {len(removed)}  (サンプルナンバー {removed['sample number'].min()}〜{removed['sample number'].max()})")
print(f"除外後件数: {len(cleaned_beisugi_train)}")
cleaned_beisugi_train.head()

除外件数: 52  (サンプルナンバー 1556〜1607)
除外後件数: 1270


,sample number,species number,樹種,含水率,9993.76781,9989.9107,9986.05359,9982.19648,9978.33937,9974.48227,...,4034.53536,4030.67826,4026.82115,4022.96404,4019.10693,4015.24982,4011.39271,4007.5356,4003.6785,3999.82139
0,1,1,イチョウ,216.129032,0.41485,0.41465,0.41463,0.41476,0.41481,0.41470,...,1.25104,1.24925,1.24145,1.23620,1.23384,1.22981,1.22818,1.23087,1.23354,1.23219
1,2,1,イチョウ,210.752688,0.42049,0.42040,0.42049,0.42053,0.42038,0.42010,...,1.21929,1.21611,1.21565,1.21745,1.21680,1.21205,1.21074,1.21508,1.21901,1.21846
2,3,1,イチョウ,205.913979,0.41040,0.41045,0.41047,0.41028,0.41000,0.40989,...,1.17471,1.17147,1.16611,1.16633,1.16998,1.16955,1.16200,1.15341,1.15139,1.15403
3,4,1,イチョウ,201.075269,0.40080,0.40061,0.40031,0.40008,0.39993,0.39991,...,1.11271,1.11339,1.11115,1.10965,1.11017,1.11165,1.11426,1.11787,1.11849,1.11328
4,5,1,イチョウ,196.236559,0.38792,0.38814,0.38825,0.38817,0.38798,0.38778,...,1.05617,1.05596,1.05760,1.06014,1.06240,1.06566,1.06962,1.07056,1.06459,1.05612


## 保存

In [3]:
output_path = "../data/interim/cleaned_beisugi_train.csv"
cleaned_beisugi_train.to_csv(output_path, index=False, encoding="shift-jis")
print(f"保存完了: {output_path}")

保存完了: ../data/interim/cleaned_beisugi_train.csv


---

## トチ・チェリー異常サンプルの除去

`cleaned_beisugi_train` からさらに以下を除外する。

- サンプルナンバー **977〜979**（チェリー：乾燥初期の異常点）
- サンプルナンバー **1148〜1247**（トチ：乾燥末期の跳ね上がり）

In [4]:
remove_ranges = [
    (977, 979),    # チェリー 異常点
    (1148, 1247),  # トチ 異常点
]

mask2 = ~cleaned_beisugi_train["sample number"].apply(
    lambda s: any(lo <= s <= hi for lo, hi in remove_ranges)
)
cleaned_three_species_train = cleaned_beisugi_train[mask2].reset_index(drop=True)

removed2 = cleaned_beisugi_train[~mask2]
print(f"除外件数: {len(removed2)}")
for lo, hi in remove_ranges:
    n = ((removed2["sample number"] >= lo) & (removed2["sample number"] <= hi)).sum()
    print(f"  サンプルナンバー {lo}〜{hi}: {n}件")
print(f"除外後件数: {len(cleaned_three_species_train)}")
cleaned_three_species_train.head()

除外件数: 103
  サンプルナンバー 977〜979: 3件
  サンプルナンバー 1148〜1247: 100件
除外後件数: 1167


,sample number,species number,樹種,含水率,9993.76781,9989.9107,9986.05359,9982.19648,9978.33937,9974.48227,...,4034.53536,4030.67826,4026.82115,4022.96404,4019.10693,4015.24982,4011.39271,4007.5356,4003.6785,3999.82139
0,1,1,イチョウ,216.129032,0.41485,0.41465,0.41463,0.41476,0.41481,0.41470,...,1.25104,1.24925,1.24145,1.23620,1.23384,1.22981,1.22818,1.23087,1.23354,1.23219
1,2,1,イチョウ,210.752688,0.42049,0.42040,0.42049,0.42053,0.42038,0.42010,...,1.21929,1.21611,1.21565,1.21745,1.21680,1.21205,1.21074,1.21508,1.21901,1.21846
2,3,1,イチョウ,205.913979,0.41040,0.41045,0.41047,0.41028,0.41000,0.40989,...,1.17471,1.17147,1.16611,1.16633,1.16998,1.16955,1.16200,1.15341,1.15139,1.15403
3,4,1,イチョウ,201.075269,0.40080,0.40061,0.40031,0.40008,0.39993,0.39991,...,1.11271,1.11339,1.11115,1.10965,1.11017,1.11165,1.11426,1.11787,1.11849,1.11328
4,5,1,イチョウ,196.236559,0.38792,0.38814,0.38825,0.38817,0.38798,0.38778,...,1.05617,1.05596,1.05760,1.06014,1.06240,1.06566,1.06962,1.07056,1.06459,1.05612


In [5]:
output_path2 = "../data/interim/cleaned_three_species_train.csv"
cleaned_three_species_train.to_csv(output_path2, index=False, encoding="shift-jis")
print(f"保存完了: {output_path2}")

保存完了: ../data/interim/cleaned_three_species_train.csv
